# Conversion of DinoV3 model to tflite

Install torch, tensorflow and onnx Libraries
Onnx is necessary as there is no direct conversion between pytorch and tflite

In [ ]:
import sys
!{sys.executable} -m pip install numpy==2.0.2 transformers torch==2.10.0 torchmetrics onnx==1.19.1 onnxconverter_common onnxscript tensorflow onnx-tf onnxruntime onnx2tf onnxsim ai_edge_litert sng4onnx onnx_graphsurgeon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 7.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of onnx-tf to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of onnx2tf to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.3/222.3 kB 13.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.3/222.3 kB 18.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.3/220.3 kB 15.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.3/220.3 kB 13.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.3/220.3 kB 16.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.2/220.2 kB 16.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 kB 18.2 MB/s eta 0:00:00
INFO: pip is still looking at multiple versions of onnx2tf to de

Import necessary libraries



In [ ]:
from __future__ import annotations

import torch
import torch.nn as nn
from google.colab import userdata
import os

# Load and create Image Encoder model

Create Image Encoder Model Class

In [ ]:
import torch.nn.functional as F

class DinoV3FeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        # DINOv2 hat einen eingebauten class token (CLS) — nutze ihn direkt
        features = self.model.forward_features(x)
        cls_token = features["x_norm_clstoken"]  # (B, 384) — bereits normalisiert
        return cls_token

Load Dinov3 model(Credits to https://github.com/IoT-gamer/segment-anything-dinov3-onnx/blob/main/notebooks/dinov3_tflite_export.ipynb)

In [ ]:
MODEL_DINOV3_VITS = "dinov3_vits16"
MODEL_NAME = MODEL_DINOV3_VITS

DEVICE = "cpu" # Select cuda or cpu

# Set the path to your local weights file. If `None`, it will download from the URL.
# For example: WEIGHTS_LOCAL_PATH = "/path/to/dinov3_vits16.pt"

# WEIGHTS_LOCAL_PATH = None
WEIGHTS_LOCAL_PATH = "dinov3_vits16_pretrain_lvd1689m-08c60483.pth"

if WEIGHTS_LOCAL_PATH and os.path.exists(WEIGHTS_LOCAL_PATH):
    print(f"Loading weights from local path: {WEIGHTS_LOCAL_PATH}")
    weights_source = WEIGHTS_LOCAL_PATH
else:
    print("Loading weights from remote URL.")
    # Add a secret named `dinov3_vits16` in Colab with the URL as the value
    try:
        WEIGHTS_URL = userdata.get('dinov3_vits16')
        weights_source = WEIGHTS_URL
    except Exception as e:
        print(f"Could not retrieve weights URL from Colab secrets. Please set WEIGHTS_LOCAL_PATH. Error: {e}")
        weights_source = None # Will cause an error if not set

# Load the model
dinov3_model = torch.hub.load(
    repo_or_dir="facebookresearch/dinov3",
    model=MODEL_NAME,
    source="github",
    weights=weights_source
)

Loading weights from local path: dinov3_vits16_pretrain_lvd1689m-08c60483.pth
Downloading: "https://github.com/facebookresearch/dinov3/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "file:///content/dinov3_vits16_pretrain_lvd1689m-08c60483.pth" to /root/.cache/torch/hub/checkpoints/dinov3_vits16_pretrain_lvd1689m-08c60483.pth


100%|██████████| 82.5M/82.5M [00:00<00:00, 1.06GB/s]


Create Dino Image encoder

In [ ]:
encoder = DinoV3FeatureExtractor(dinov3_model)
encoder.eval()

DinoV3FeatureExtractor(
  (model): DinoVisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (rope_embed): RopePositionEmbedding()
    (blocks): ModuleList(
      (0-11): 12 x SelfAttentionBlock(
        (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (attn): SelfAttention(
          (qkv): LinearKMaskedBias(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0

Check if model works as intended

In [ ]:
dummy_input = torch.randn(1, 3, 98, 98)

with torch.no_grad():
    output_shape = encoder(dummy_input).shape
    expected_shape = (1, 384)
    print(f"PyTorch output shape: {output_shape}")
    assert output_shape == expected_shape, f"Unexpected output shape! Got {output_shape}, expected {expected_shape}"

PyTorch output shape: torch.Size([1, 384])


Export to onnx

In [ ]:
torch.onnx.export(
    encoder,
    dummy_input,
    "dino_image_encoder.onnx",
    opset_version=18,
    input_names=["pixel_values"],
    output_names=["embeddings"],
    dynamic_axes=None, # fixed batch size 1 for on-device use
    dynamo=False, # use dynamic_axes instead of dynamic_shapes
    do_constant_folding=True,
)
print("ONNX export done.")

/tmp/ipykernel_1095/1279171570.py:1: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/root/.cache/torch/hub/facebookresearch_dinov3_main/dinov3/layers/block.py:133: TracerWarning: Converting a tensor to a Python integer might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  sample_subset_sizes = [max(int(b * (1 - self.sample_drop_ratio)), 1) for b in b_list]
/root/.cache/torch/hub/facebookresearch_dinov3_main/dinov3/layers/attention.py:76: TracerWarning: Converting a tensor to a Python

ONNX export done.


# Conversion to tflite

Ensure the model is fully simplified

In [ ]:
import onnx
from onnxsim import simplify
model = onnx.load("dino_image_encoder.onnx")
model_simp, check = simplify(model)
onnx.save(model_simp, "dino_image_encoder_simplified.onnx")

assert check, "Simplified ONNX model could not be validated"

In [ ]:
import onnx2tf

OUTPUT_DIR = "dino_tflite_output"
ONNX_PATH = "dino_image_encoder_simplified.onnx"

onnx2tf.convert(
    input_onnx_file_path=ONNX_PATH,
    output_folder_path=OUTPUT_DIR,
    not_use_onnxsim=True, # Simplification was already done
    non_verbose=False,
    replace_to_pseudo_operators=["erf"] # Try to avoid Flex Operations(not supported by tflite flutter)
)

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
INFO: 199 / 656
INFO: onnx_op_type: Transpose onnx_op_name: /blocks.3/attn/Transpose_3
INFO:  input_name.1: /blocks.3/attn/Concat_4_output_0 shape: [1, 6, 41, 64] dtype: float32
INFO:  output_name.1: /blocks.3/attn/Transpose_3_output_0 shape: [1, 6, 64, 41] dtype: float32
INFO: tf_op_type: transpose_v2
INFO:  input.1.a: name: tf.concat_129/concat:0 shape: (1, 6, 41, 64) dtype: <dtype: 'float32'> 
INFO:  input.2.perm: val: [0, 1, 3, 2] 
INFO:  output.1.output: name: tf.compat.v1.transpose_365/transpose:0 shape: (1, 6, 64, 41) dtype: <dtype: 'float32'> 

INFO: 200 / 656
INFO: onnx_op_type: Mul onnx_op_name: /blocks.3/attn/Mul_9
INFO:  input_name.1: /blocks.3/attn/Transpose_3_output_0 shape: [1, 6, 64, 41] dtype: float32
INFO:  input_name.2: /blocks.0/attn/Sqrt_1_output_0 shape: (1,) dtype: float32
INFO:  output_name.1: /blocks.3/attn/Mul_9_output_0 shape: [1, 6, 64, 41] dtype: float32
INFO: tf_op_type: multiply
INFO:  inp

Convert .pb to .tflite, Specify supported operations

In [ ]:
import tensorflow as tf
from pathlib import Path

# OUTPUT_DIR already contains the SavedModel generated by onnx2tf.convert
converter = tf.lite.TFLiteConverter.from_saved_model(OUTPUT_DIR)
converter.target_spec.supported_ops = [
  tf.lite.OpsSet.TFLITE_BUILTINS,# enable TensorFlow Lite ops.
  tf.lite.OpsSet.SELECT_TF_OPS # enable TensorFlow ops.
]
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter._experimental_lower_to_version = 3
tflite_model = converter.convert()

TFLITE_PATH = Path(OUTPUT_DIR) / "dino_image_encoder_select_tf_ops.tflite"
open(TFLITE_PATH, "wb").write(tflite_model)
print(f"New TFLite model saved to: {TFLITE_PATH}")

New TFLite model saved to: dino_tflite_output/dino_image_encoder_select_tf_ops.tflite


Get Operations used by model

In [ ]:
from tensorflow.lite.python import schema_py_generated as schema_fb

def get_builtin_operator_name_robust(builtin_code):
    """
    Robustly gets the name of a TFLite BuiltinOperator from its integer code,
    handling cases where EnumToName might be missing or the schema structure varies.
    """
    # Attempt to use EnumToName if it exists (for compatibility)
    if hasattr(schema_fb.BuiltinOperator, 'EnumToName'):
        return schema_fb.BuiltinOperator.EnumToName(builtin_code)

    # Fallback: iterate through attributes to find the name
    for name in dir(schema_fb.BuiltinOperator):
        if not name.startswith('_'): # Filter out internal attributes
            value = getattr(schema_fb.BuiltinOperator, name)
            if value == builtin_code:
                return name
    return f"UNKNOWN_BUILTIN_OP_{builtin_code}"

def get_tflite_operations(model_path):
    with open(model_path, 'rb') as f:
        model_buffer = f.read()
    model = schema_fb.Model.GetRootAsModel(model_buffer, 0)

    op_codes = set()
    for subgraph_idx in range(model.SubgraphsLength()):
        subgraph = model.Subgraphs(subgraph_idx)
        for op_idx in range(subgraph.OperatorsLength()):
            op = subgraph.Operators(op_idx)
            op_code_idx = op.OpcodeIndex()
            builtin_code = model.OperatorCodes(op_code_idx).BuiltinCode()

            if builtin_code == schema_fb.BuiltinOperator.CUSTOM:
                custom_code = model.OperatorCodes(op_code_idx).CustomCode().decode('utf-8')
                op_codes.add(f"CUSTOM: {custom_code}")
            else:
                builtin_code_name = get_builtin_operator_name_robust(builtin_code)
                op_codes.add(builtin_code_name)
    return sorted(list(op_codes))

print(f"Operations in TFLite model: {get_tflite_operations(TFLITE_PATH)}")

Operations in TFLite model: ['ABS', 'ADD', 'BATCH_MATMUL', 'CONCATENATION', 'CONV_2D', 'DIV', 'EXP', 'FULLY_CONNECTED', 'GATHER', 'MEAN', 'MUL', 'NEG', 'RESHAPE', 'RSQRT', 'SOFTMAX', 'SQUARED_DIFFERENCE', 'STRIDED_SLICE', 'SUB', 'TRANSPOSE']


Check dimensions of converted model

In [ ]:
interpreter = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"  Input  shape  : {input_details[0]['shape']}")   # [1, 224, 224, 3]
print(f"  Output shape  : {output_details[0]['shape']}")  # [1, 512]

  Input  shape  : [ 1 98 98  3]
  Output shape  : [  1 384]


Run with a random numpy input

In [ ]:
import numpy as np
test_input = np.random.rand(1, 98, 98, 3).astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], test_input)
interpreter.invoke()
result = interpreter.get_tensor(output_details[0]['index'])
print(f"  Inference output shape: {result.shape}")
print(f"  Inference output: {result}")

  Inference output shape: (1, 384)
  Inference output: [[ 0.08602294  0.13880052  0.10431373 -0.36329022  0.5721819  -0.1520582
   0.14187036  0.02265012  0.12554623 -0.12628831 -0.04123566 -0.45183676
  -0.6461412  -0.8581778   0.14015175  0.46346214  0.31412467 -0.20756258
  -0.510144    0.17107809 -0.23544794  0.12456904 -0.04495357 -0.4672766
  -0.5529287  -0.0168229   0.23078835  0.0138333  -0.2605317   0.7411964
  -0.3508287   0.6829246  -0.49685782  0.46259475  0.36269838  0.40219665
   0.06659426  0.311382    0.684577    0.90322155 -0.17442222  0.59720975
  -0.17676795 -0.36409128  0.6002533  -0.1125031   0.5295466  -0.49570873
   0.03304385  0.2168386  -0.1630245   0.00910649 -0.1937046   0.25047776
  -0.37299266  0.17793399 -0.29736316 -1.0433682   0.17674184 -0.00512929
   0.64753234  0.08467668 -0.19511105  0.20545807  0.5915656   0.0248833
  -0.5497572   0.00934374  0.83425057  0.5347405   1.1441991   0.25970978
  -0.0769688   0.91810286  0.39492896 -0.0311788  -0.05155827